<a href="https://colab.research.google.com/github/ameerabdulfatah/Customer-Churn-Analysis-From-Python-to-Power-BI/blob/main/%20code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install if needed (Colab/local)
# !pip install lifelines

import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

def preprocess_and_km_by_contract_plot(file_path="/content/Churn Data.csv"):
    # 1. Load data
    df = pd.read_csv(file_path)

    # 2. Minimal cleaning
    df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce")
    df["Total Charges"] = df["Total Charges"].fillna(0)
    df["Internet Type"] = df["Internet Type"].fillna("No Internet")
    df["Offer"] = df["Offer"].fillna("No Offer")
    if "Churn Category" in df.columns:
        df["Churn Category"] = df["Churn Category"].fillna("Not Churned")
    if "Churn Reason" in df.columns:
        df["Churn Reason"] = df["Churn Reason"].fillna("Not Churned")

    drop_cols = [
        "Customer ID", "Churn Category", "Churn Reason", "Customer Status",
        "Quarter", "Country", "State", "City", "Zip Code", "Lat Long",
        "Latitude", "Longitude"
    ]
    df = df.drop(columns=drop_cols, errors="ignore")

    # 3. Define X, y and preprocessing pipeline
    target = "Churn"
    X = df.drop(target, axis=1)
    y = df[target]

    numeric_features = [
        "Age", "Avg Monthly GB Download", "Avg Monthly Long Distance Charges",
        "Churn Score", "CLTV", "Monthly Charge", "Number of Dependents",
        "Number of Referrals", "Population", "Tenure in Months", "Total Charges",
        "Total Extra Data Charges", "Total Long Distance Charges", "Total Refunds",
        "Total Revenue", "Satisfaction Score"
    ]
    categorical_features = ["Contract", "Gender", "Internet Type", "Offer", "Payment Method"]
    binary_features = [
        "Dependents", "Device Protection Plan", "Internet Service", "Married",
        "Multiple Lines", "Online Backup", "Online Security", "Paperless Billing",
        "Partner", "Phone Service", "Premium Tech Support", "Referred a Friend",
        "Senior Citizen", "Streaming Movies", "Streaming Music", "Streaming TV",
        "Under 30", "Unlimited Data"
    ]

    numeric_features = [c for c in numeric_features if c in X.columns]
    categorical_features = [c for c in categorical_features if c in X.columns]
    binary_features = [c for c in binary_features if c in X.columns]

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])
    binary_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
            ("bin", binary_transformer, binary_features),
        ],
        remainder="drop",
    )

    X_processed = preprocessor.fit_transform(X)

    # 4. Kaplan–Meier plots by contract type only
    plt.figure(figsize=(10, 6))
    ax = plt.subplot(111)

    for contract_type in df["Contract"].dropna().unique():
        df_seg = df[df["Contract"] == contract_type]
        kmf_seg = KaplanMeierFitter()
        kmf_seg.fit(
            durations=df_seg["Tenure in Months"],
            event_observed=df_seg["Churn"],
            label=str(contract_type)
        )
        kmf_seg.plot_survival_function(ax=ax, ci_show=False)

    plt.title("Kaplan-Meier by Contract Type")
    plt.xlabel("Tenure (months)")
    plt.ylabel("Survival Probability")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return X_processed, y, preprocessor

# Run everything
if __name__ == "__main__":
    X_processed, y, preprocessor = preprocess_and_km_by_contract_plot()
